# HW Part 1: Value iteration convergence (see seminar 2)

### Find an MDP for which value iteration takes long to converge  (0.1 pts)

When we ran value iteration on the small frozen lake problem, the last iteration where an action changed was iteration 6--i.e., value iteration computed the optimal policy at iteration 6. Are there any guarantees regarding how many iterations it'll take value iteration to compute the optimal policy? There are no such guarantees without additional assumptions--we can construct the MDP in such a way that the greedy policy will change after arbitrarily many iterations.

Your task: define an MDP with at most 3 states and 2 actions, such that when you run value iteration, the optimal action changes at iteration >= 50. Use discount=0.95. (However, note that the discount doesn't matter here--you can construct an appropriate MDP with any discount.)

Note: value function must change at least once after iteration >=50, not necessarily change on every iteration till >=50.

In [180]:
from mdp import MDP
from numpy import random

transition_probs = {
    "s0": {"a0": {"s1": 1.0}, "a1": {"s2": 1.0}},
    "s1": {"a0": {"s1": 1.0}, "a1": {"s2": 1.0}},
    "s2": {},
}

rewards = {
    "s0": {"a0": {"s1": 0.0}, "a1": {"s2": 18.0}},
    "s1": {"a0": {"s1": 1.0}, "a1": {"s2": 0.0}},
}



mdp = MDP(transition_probs, rewards, initial_state=random.choice(tuple(transition_probs.keys())))
# Feel free to change the initial_state

In [181]:
import numpy as np


def get_optimal_action(mdp: MDP, state_values: dict, state, gamma: float = 0.9):
    if mdp.is_terminal(state):
        return None

    best_action = None
    best_q = -np.inf

    for action in mdp.get_possible_actions(state):
        q = 0.0
        for next_state in mdp.get_next_states(state, action):
            prob = mdp.get_transition_prob(state, action, next_state)
            reward = mdp.get_reward(state, action, next_state)
            q += prob * (reward + gamma * state_values[next_state])

        if q > best_q:
            best_q = q
            best_action = action

    return best_action


def get_new_state_value(mdp: MDP, state_values: dict, state: str, gamma: float):
    """Computes next V(s) as in formula above. Please do not change state_values in process."""
    if mdp.is_terminal(state):
        return 0

    max_value = -np.inf
    for action in mdp.get_possible_actions(state):
        value = 0
        for next_state in mdp.get_next_states(state, action):
            r = mdp.get_reward(state, action, next_state)
            next_v = state_values[next_state]
            prob = mdp.get_transition_prob(state, action, next_state)
            value += prob * (r + gamma * next_v)
        if value > max_value:
            max_value = value

    return max_value


def value_iteration(mdp: MDP, state_values: dict = None, gamma: float = 0.9, num_iter=1000, min_difference=1e-5):
    """performs num_iter value iteration steps starting from state_values. Same as before but in a function"""
    state_values = state_values or {s: 0 for s in mdp.get_all_states()}
    for i in range(num_iter):
        # Compute new state values using the functions you defined above. It must be a dict {state : new_V(state)}
        new_state_values = {s: get_new_state_value(mdp, state_values, s, gamma) for s in mdp.get_all_states()}

        assert isinstance(new_state_values, dict)

        # Compute difference
        diff = max(abs(new_state_values[s] - state_values[s]) for s in mdp.get_all_states())

        print("iter %4i   |   diff: %6.5f   |   V(start): %.3f " % (i, diff, new_state_values[mdp._initial_state]))

        state_values = new_state_values
        if diff < min_difference:
            break

    return state_values


In [183]:
gamma = 0.95
state_values = {s: 0 for s in mdp.get_all_states()}
policy = np.array([get_optimal_action(mdp, state_values, state, gamma) for state in sorted(mdp.get_all_states())])

for i in range(100):
    print("after iteration %i" % i)
    state_values = value_iteration(mdp, state_values, gamma=gamma, num_iter=1)

    new_policy = np.array([get_optimal_action(mdp, state_values, state, gamma) for state in sorted(mdp.get_all_states())])

    n_changes = (policy != new_policy).sum()
    print("N actions changed = %i \n" % n_changes)
    policy = new_policy

# please ignore iter 0 at each step

after iteration 0
iter    0   |   diff: 18.00000   |   V(start): 0.000 
N actions changed = 0 

after iteration 1
iter    0   |   diff: 0.95000   |   V(start): 0.000 
N actions changed = 0 

after iteration 2
iter    0   |   diff: 0.90250   |   V(start): 0.000 
N actions changed = 0 

after iteration 3
iter    0   |   diff: 0.85737   |   V(start): 0.000 
N actions changed = 0 

after iteration 4
iter    0   |   diff: 0.81451   |   V(start): 0.000 
N actions changed = 0 

after iteration 5
iter    0   |   diff: 0.77378   |   V(start): 0.000 
N actions changed = 0 

after iteration 6
iter    0   |   diff: 0.73509   |   V(start): 0.000 
N actions changed = 0 

after iteration 7
iter    0   |   diff: 0.69834   |   V(start): 0.000 
N actions changed = 0 

after iteration 8
iter    0   |   diff: 0.66342   |   V(start): 0.000 
N actions changed = 0 

after iteration 9
iter    0   |   diff: 0.63025   |   V(start): 0.000 
N actions changed = 0 

after iteration 10
iter    0   |   diff: 0.59874 

```
after iteration 57
iter    0   |   diff: 0.05373   |   V(start): 0.000 
N actions changed = 1
```

### Value iteration convervence proof (0.1 pts)
**Note:** Assume that $\mathcal{S}, \mathcal{A}$ are finite.

Update of value function in value iteration can be rewritten in a form of Bellman operator:

$$(TV)(s) = \max_{a \in \mathcal{A}}\mathbb{E}\left[ r_{t+1} + \gamma V(s_{t+1}) | s_t = s, a_t = a\right]$$

Value iteration algorithm with Bellman operator:

---
&nbsp;&nbsp; Initialize $V_0$

&nbsp;&nbsp; **for** $k = 0,1,2,...$ **do**

&nbsp;&nbsp;&nbsp;&nbsp; $V_{k+1} \leftarrow TV_k$

&nbsp;&nbsp;**end for**

---

In [lecture](https://docs.google.com/presentation/d/1lz2oIUTvd2MHWKEQSH8hquS66oe4MZ_eRvVViZs2uuE/edit#slide=id.g4fd6bae29e_2_4) we established contraction property of bellman operator:

$$
||TV - TU||_{\infty} \le \gamma ||V - U||_{\infty}
$$

For all $V, U$

Using contraction property of Bellman operator, Banach fixed-point theorem and Bellman equations prove that value function converges to $V^*$ in value iterateion

### Proof

Given the contraction property, and the fact that $TV^* = V^*$, we have:
$$
\lim_{t \to \infty}\|T^tV - V^*\|_{\infty} = \lim_{t \to \infty}\|T^tV - T^tV^*\|_{\infty} \leq \gamma^t \|V - V^*\|_{\infty} = 0
$$

### Asynchronious value iteration (0.2 pts)

Consider the following algorithm:

---

Initialize $V_0$

**for** $k = 0,1,2,...$ **do**

&nbsp;&nbsp;&nbsp;&nbsp; Select some state $s_k \in \mathcal{S}$    

&nbsp;&nbsp;&nbsp;&nbsp; $V(s_k) := (TV)(s_k)$

**end for**

---


Note that unlike common value iteration, here we update only a single state at a time.

**Homework.** Prove the following proposition:

If for all $s \in \mathcal{S}$, $s$ appears in the sequence $(s_0, s_1, ...)$ infinitely often, then $V$ converges to $V*$

### Proof
Using the property that for any $s$:
$$
TV(s) - V^*(s) \le \gamma (V(s) - V^*(s))
$$
The value of $\|TV - V^*\|_{\infty}$ does not increase over time. It decreases every time we see all $s \in \mathcal{S}$ in a contiguous chunk of sequence. Since each $s$ appears infinitely often, the value of $\|TV - V^*\|_{\infty}$ will eventually converge to 0.

# HW Part 2: Policy iteration

## Policy iteration implementateion (0.3 pts)

Let's implement exact policy iteration (PI), which has the following pseudocode:

---
Initialize $\pi_0$   `// random or fixed action`

For $n=0, 1, 2, \dots$
- Compute the state-value function $V^{\pi_{n}}$
- Using $V^{\pi_{n}}$, compute the state-action-value function $Q^{\pi_{n}}$
- Compute new policy $\pi_{n+1}(s) = \operatorname*{argmax}_a Q^{\pi_{n}}(s,a)$
---

Unlike VI, policy iteration has to maintain a policy - chosen actions from all states - and estimate $V^{\pi_{n}}$ based on this policy. It only changes policy once values converged.


Below are a few helpers that you may or may not use in your implementation.

In [184]:
transition_probs = {
    's0': {
        'a0': {'s0': 0.5, 's2': 0.5},
        'a1': {'s2': 1}
    },
    's1': {
        'a0': {'s0': 0.7, 's1': 0.1, 's2': 0.2},
        'a1': {'s1': 0.95, 's2': 0.05}
    },
    's2': {
        'a0': {'s0': 0.4, 's1': 0.6},
        'a1': {'s0': 0.3, 's1': 0.3, 's2': 0.4}
    }
}
rewards = {
    's1': {'a0': {'s0': +5}},
    's2': {'a1': {'s0': -1}}
}

mdp = MDP(transition_probs, rewards, initial_state='s0')

Let's write a function called `compute_vpi` that computes the state-value function $V^{\pi}$ for an arbitrary policy $\pi$.

Unlike VI, this time you must find the exact solution, not just a single iteration.

Recall that $V^{\pi}$ satisfies the following linear equation:
$$V^{\pi}(s) = \sum_{s'} P(s,\pi(s),s')[ R(s,\pi(s),s') + \gamma V^{\pi}(s')]$$

You'll have to solve a linear system in your code. (Find an exact solution, e.g., with `np.linalg.solve`.)

In [185]:
def compute_vpi(mdp: MDP, policy: dict, gamma: float):
    """
    Computes V^pi(s) FOR ALL STATES under given policy.
    :param policy: a dict of currently chosen actions {s : a}
    :returns: a dict {state : V^pi(state) for all states}
    """
    states = list(mdp.get_all_states())
    state2idx = {state: i for i, state in enumerate(states)}
    num_states = len(states)

    nonterminal_states = [s for s in states if not mdp.is_terminal(s)]
    assert set(policy.keys()) == set(nonterminal_states)
    assert all(policy[s] in mdp.get_possible_actions(s) for s in nonterminal_states)

    P_pi = np.zeros((num_states, num_states))
    r_pi = np.zeros(num_states)

    for state in states:
        if mdp.is_terminal(state):
            continue

        i = state2idx[state]
        action = policy[state]

        for next_state in mdp.get_next_states(state, action):
            j = state2idx[next_state]
            prob = mdp.get_transition_prob(state, action, next_state)
            reward = mdp.get_reward(state, action, next_state)

            P_pi[i, j] = prob
            r_pi[i] += prob * reward

    A = np.eye(num_states) - gamma * P_pi
    v = np.linalg.solve(A, r_pi)

    return {state: v[state2idx[state]] for state in states}



In [186]:
test_policy = {s: np.random.choice(mdp.get_possible_actions(s)) for s in mdp.get_all_states()}
new_vpi = compute_vpi(mdp, test_policy, gamma)

print(new_vpi)

assert type(new_vpi) is dict, "compute_vpi must return a dict {state : V^pi(state) for all states}"

{'s0': np.float64(12.657572671263736), 's1': np.float64(16.105387941829576), 's2': np.float64(13.989948741923078)}


Once we've got new state values, it's time to update our policy.

In [187]:
def compute_new_policy(mdp: MDP, vpi: dict, gamma: float):
    """
    Computes new policy as argmax of state values
    :param vpi: a dict {state : V^pi(state) for all states}
    :returns: a dict {state : optimal action for all states}
    """
    return {s: get_optimal_action(mdp, vpi, s, gamma) for s in mdp.get_all_states() if not mdp.is_terminal(s)}

In [188]:
new_policy = compute_new_policy(mdp, new_vpi, gamma)

print(new_policy)

assert type(new_policy) is dict, "compute_new_policy must return a dict {state : optimal action for all states}"

{'s0': 'a1', 's1': 'a0', 's2': 'a0'}


__Main loop__

In [189]:
def policy_iteration(mdp: MDP, policy: dict = None, gamma: float = 0.9, num_iter: int = 1000, min_difference: float = 1e-5):
    """
    Run the policy iteration loop for num_iter iterations or till difference between V(s) is below min_difference.
    If policy is not given, initialize it at random.
    """
    nonterminal_states = [s for s in mdp.get_all_states() if not mdp.is_terminal(s)]

    if policy is None:
        policy = {s: np.random.choice(mdp.get_possible_actions(s)) for s in nonterminal_states}

    vpi = compute_vpi(mdp, policy, gamma)
    for i in range(num_iter):
        new_policy = compute_new_policy(mdp, vpi, gamma)
        new_vpi = compute_vpi(mdp, new_policy, gamma)
        delta = max(abs(new_vpi[s] - vpi[s]) for s in nonterminal_states)
        n_changed = sum(policy[s] != new_policy[s] for s in nonterminal_states)
        print(f"Iteration {i}, delta: {delta}, changed actions: {n_changed}")

        if n_changed == 0 or delta < min_difference:
            return vpi, policy

        policy = new_policy
        vpi = new_vpi

    return compute_vpi(mdp, policy, gamma), policy

__Your PI Results__

In [190]:
from mdp import FrozenLakeEnv

env_small = FrozenLakeEnv(map_name="4x4")
vpi_small, policy_small = policy_iteration(env_small, gamma=0.99)

Iteration 0, delta: 0.7569224460759791, changed actions: 7
Iteration 1, delta: 0.11539975524538704, changed actions: 4
Iteration 2, delta: 0.012359223966732613, changed actions: 1
Iteration 3, delta: 0.0, changed actions: 0


In [191]:
env_large = FrozenLakeEnv(map_name="8x8")
vpi_large, policy_large = policy_iteration(env_large, gamma=0.99)

Iteration 0, delta: 0.7869010356799576, changed actions: 43
Iteration 1, delta: 0.21693042422067232, changed actions: 17
Iteration 2, delta: 0.06602125167530998, changed actions: 8
Iteration 3, delta: 0.008938392853775845, changed actions: 4
Iteration 4, delta: 0.0, changed actions: 0


In [192]:
def print_frozenlake_values_and_policy(env, values, policy):
    arrows = {
        "left": "←",
        "right": "→",
        "up": "↑",
        "down": "↓",
    }

    nrow, ncol = env.desc.shape
    cell_width = 16

    def format_cell(state):
        i, j = state
        cell_type = env.desc[i, j]

        if cell_type in ("H", "G"):
            arrow = "·"
        else:
            arrow = arrows.get(policy.get(state), "?")

        value = values[state]
        top = f"{cell_type} {state}"
        bottom = f"V={value:7.3f} {arrow}"
        return top, bottom

    sep = "+" + "+".join(["-" * cell_width for _ in range(ncol)]) + "+"

    for i in range(nrow):
        print(sep)

        top_row = []
        bottom_row = []
        for j in range(ncol):
            top, bottom = format_cell((i, j))
            top_row.append(f"{top:<{cell_width}}")
            bottom_row.append(f"{bottom:<{cell_width}}")

        print("|" + "|".join(top_row) + "|")
        print("|" + "|".join(bottom_row) + "|")

    print(sep)

print_frozenlake_values_and_policy(env_small, vpi_small, policy_small)

+----------------+----------------+----------------+----------------+
|S (0, 0)        |F (0, 1)        |F (0, 2)        |F (0, 3)        |
|V=  0.716 ↓     |V=  0.676 ↑     |V=  0.705 ↓     |V=  0.640 ↑     |
+----------------+----------------+----------------+----------------+
|F (1, 0)        |H (1, 1)        |F (1, 2)        |H (1, 3)        |
|V=  0.730 ←     |V=  0.000 ·     |V=  0.725 ↓     |V=  0.000 ·     |
+----------------+----------------+----------------+----------------+
|F (2, 0)        |F (2, 1)        |F (2, 2)        |H (2, 3)        |
|V=  0.818 →     |V=  0.942 ↓     |V=  0.916 ←     |V=  0.000 ·     |
+----------------+----------------+----------------+----------------+
|H (3, 0)        |F (3, 1)        |F (3, 2)        |G (3, 3)        |
|V=  0.000 ·     |V=  0.972 →     |V=  0.988 →     |V=  0.000 ·     |
+----------------+----------------+----------------+----------------+


In [193]:
print_frozenlake_values_and_policy(env_large, vpi_large, policy_large)

+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+
|S (0, 0)        |F (0, 1)        |F (0, 2)        |F (0, 3)        |F (0, 4)        |F (0, 5)        |F (0, 6)        |F (0, 7)        |
|V=  0.672 →     |V=  0.682 →     |V=  0.691 →     |V=  0.701 →     |V=  0.711 →     |V=  0.719 →     |V=  0.728 ↓     |V=  0.735 ↓     |
+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+
|F (1, 0)        |F (1, 1)        |F (1, 2)        |F (1, 3)        |F (1, 4)        |F (1, 5)        |F (1, 6)        |F (1, 7)        |
|V=  0.666 →     |V=  0.674 →     |V=  0.683 ↑     |V=  0.694 ↑     |V=  0.715 →     |V=  0.726 →     |V=  0.737 ↓     |V=  0.746 ↓     |
+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+
|F (2, 0)        |F (2, 1)        

## Policy iteration convergence (0.3 pts)

**Note:** Assume that $\mathcal{S}, \mathcal{A}$ are finite.

We can define another Bellman operator:

$$(T_{\pi}V)(s) = \mathbb{E}_{r, s'|s, a = \pi(s)}\left[r + \gamma V(s')\right]$$

And rewrite policy iteration algorithm in operator form:


---

Initialize $\pi_0$

**for** $k = 0,1,2,...$ **do**

&nbsp;&nbsp;&nbsp;&nbsp; Solve $V_k = T_{\pi_k}V_k$   

&nbsp;&nbsp;&nbsp;&nbsp; Select $\pi_{k+1}$ s.t. $T_{\pi_{k+1}}V_k = TV_k$ 

**end for**

---

To prove convergence of the algorithm we need to prove two properties: contraction an monotonicity.

#### Monotonicity (0.1 pts)

For all $V, U$ if $V(s) \le U(s)$   $\forall s \in \mathcal{S}$ then $(T_\pi V)(s) \le (T_\pi U)(s)$   $\forall s \in  \mathcal{S}$

#### Proof
We can use $a = \pi(s)$ because policy is argmax
$$
V(s) \le U(s): \forall s \in \mathcal{S} \implies \\
\mathbb{E}_{p(s^{\prime} | s, a)}[V(s^{\prime})] \le \mathbb{E}_{p(s^{\prime} | s, a)}[U(s^{\prime})] \implies \\
\mathbb{E}_{p(s^{\prime} | s, a)}[r(s, a, s^{\prime}) + \gamma V(s^{\prime})] \le \mathbb{E}_{p(s^{\prime} | s, a)}[r(s, a, s^{\prime}) + \gamma U(s^{\prime})] \iff \\
(T_\pi V)(s) \le (T_\pi U)(s)
$$

#### Contraction (0.1 pts)

$$
||T_\pi V - T_\pi U||_{\infty} \le \gamma ||V - U||_{\infty}
$$

For all $V, U$

#### Proof
Take $a = \pi(s)$, then for any $s \in \mathcal{S}$
$$
T_\pi V(s) - T_\pi U(s) = \gamma \mathbb{E}_{p(s^{\prime} | s, a)}[r(s, a, s^{\prime}) + V(s^{\prime})] - \gamma \mathbb{E}_{p(s^{\prime} | s, a)}[r(s, a, s^{\prime}) + U(s^{\prime})] = \gamma \mathbb{E}_{p(s^{\prime} | s, a)}[V(s^{\prime}) - U(s^{\prime})]
$$
take absolute values:
$$
|T_\pi V(s) - T_\pi U(s)| = |\gamma \mathbb{E}_{p(s^{\prime} | s, a)}[V(s^{\prime}) - U(s^{\prime})]| \overset{\text{Jensen}}{\le} \gamma \mathbb{E}_{p(s^{\prime} | s, a)}[|V(s^{\prime}) - U(s^{\prime})|] \le \gamma ||V - U||_{\infty}
$$
taking $s = \argmax_{s \in \mathcal{S}}|T_\pi V(s) - T_\pi U(s)|$ gives:
$$
||T_\pi V - T_\pi U||_{\infty} \le \gamma ||V - U||_{\infty}
$$
#### Convergence (0.1 pts)

Prove that there exists iteration $k_0$ such that $\pi_k = \pi^*$ for all $k \ge k_0$

#### Proof
Since $\pi_{k+1}(s) = \argmax_{a \in \mathcal{A}} \mathbb{E}_{p(s^{\prime} | s, a)}[r(s, a, s^{\prime}) + \gamma V_k(s^{\prime})]$, 
then 
$$
T_{\pi_{k+1}} V^{\pi_k} \ge T_{\pi_k} V^{\pi_k} = V^{\pi_{k}}
$$
which means that sequence
$$
V^{\pi_k} \le T_{\pi_{k+1}} V^{\pi_k} \le T_{\pi_{k+1}}^2 V^{\pi_k} \le \dots
$$
is non-decreasing. This gives
$$
V_{\pi_{k+1}} = \lim_{n \to \infty} T_{\pi_{k+1}}^n V^{\pi_k} \ge V^{\pi_k}
$$
Hence 
$$
\{V^{\pi_k}\}_{k=0}^\infty
$$
is non-decreasing. If $\pi_{k+1} \ne \pi_k$, then by the we have $V^{\pi_{k+1}} > V^{\pi_k}$ for at least one state. Since the number of deterministic policies is finite, the policy cannot change infinitely many times. Therefore, there exists $k_0$ such that $\pi_{k+1} = \pi_k$ for all $k \ge k_0$. A policy that is greedy with respect to its own value function is optimal, hence $\pi_{k_0} = \pi^*$
